# Carbon Emission Regression Model

This notebook uses the prepared model-ready CSV to train, evaluate, compare, and interpret regression models.

## Before you start

Run Carbon_Emission_Modeling.ipynb first. It creates the model-ready CSV used here. Keep both CSV files and both notebooks in the same folder.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
target_column = 'CarbonEmission'
MODEL_READY_FILE = Path('Carbon Emission - LMS Model Ready.csv')

## 1. Load the model-ready data

All predictor columns in this file are numeric. CarbonEmission is the value we want the model to predict.

In [2]:
model_ready_df = pd.read_csv(MODEL_READY_FILE)
print(f'Rows: {model_ready_df.shape[0]:,} | Columns: {model_ready_df.shape[1]}')
print(f'Missing values: {model_ready_df.isna().sum().sum()}')
display(model_ready_df.head())

Rows: 10,000 | Columns: 57
Missing values: 0


,Monthly Grocery Bill,Vehicle Monthly Distance Km,Waste Bag Weekly Count,How Long TV PC Daily Hour,How Many New Clothes Monthly,How Long Internet Daily Hour,CarbonEmission,Recycling_Glass,Recycling_Metal,Recycling_Paper,...,Frequency of Traveling by Air_never,Frequency of Traveling by Air_rarely,Frequency of Traveling by Air_very frequently,Waste Bag Size_extra large,Waste Bag Size_large,Waste Bag Size_medium,Waste Bag Size_small,Energy efficiency_No,Energy efficiency_Sometimes,Energy efficiency_Yes
0,230,210,4,7,26,1,2238,0,1,0,...,0,0,0,0,1,0,0,1,0,0
1,114,9,3,9,38,5,1892,0,1,0,...,0,1,0,1,0,0,0,1,0,0
2,138,2472,1,14,47,6,2595,0,1,0,...,1,0,0,0,0,0,1,0,1,0
3,157,74,3,20,5,7,1074,1,1,1,...,0,1,0,0,0,1,0,0,1,0
4,266,8457,1,3,5,6,4743,0,0,1,...,0,0,1,0,1,0,0,0,0,1


## 4. Split the data

We keep 20% of the data unseen until evaluation. This gives a fair estimate of performance on new households. `X` contains predictors and `y` contains the carbon-emission target.

In [3]:
X = model_ready_df.drop(columns=target_column)
y = model_ready_df[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f'Training rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Predictor columns: {X.shape[1]}')

NameError: name 'target_column' is not defined

## 5. Train and compare models

Linear Regression is the simplest baseline model. Ridge Regression is a regularized linear model and needs scaled numeric inputs. Random Forest and Gradient Boosting can capture non-linear relationships. Use the same test set for every model so the comparison is fair. Lower MAE/RMSE and higher R-squared are better.

In [ ]:
models = {
    'Linear Regression (Baseline)': LinearRegression(),
    'Ridge Regression': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Random Forest': RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': root_mean_squared_error(y_test, predictions),
        'R_squared': r2_score(y_test, predictions),
    })

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(results_df.style.format({'MAE': '{:,.2f}', 'RMSE': '{:,.2f}', 'R_squared': '{:.3f}'}))
best_model_name = results_df.loc[0, 'Model']
best_model = trained_models[best_model_name]
print(f'Best model by RMSE: {best_model_name}')

## 6. Validate the best model with cross-validation

Cross-validation trains the selected model on several different splits. It is more reliable than reporting one train/test split alone. Negative MAE is returned by scikit-learn, so we multiply by -1 to present it as a normal positive error.

In [ ]:
cv_mae = -cross_val_score(best_model, X, y, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f'3-fold MAE: {cv_mae.mean():,.2f} ± {cv_mae.std():,.2f}')

## How to use the results

Use the model comparison table to select the best model: lower MAE and RMSE are better, while higher R-squared is better. Use the feature-importance chart to discuss important predictors, but do not claim it proves cause and effect.